# Exploration du référentiel géographique

**Objectif :** construire la chaîne qui relie chaque opérateur à son territoire :

```
code postal (annuaire) → commune (La Poste) → département → région (INSEE COG)
                                            ↘ population (INSEE)
                                            ↘ exploitations agricoles (Agreste)
                                            ↘ contours pour la carte (GeoJSON)
```

C'est la future dimension `dim_geographie` du modèle en étoile. Une seule erreur de code dans cette chaîne
et des opérateurs disparaissent des cartes : on vérifie donc que **tous les maillons utilisent les mêmes codes**.

| Source | Fichier |
|---|---|
| La Poste, base officielle des codes postaux | `laposte_codes_postaux/2026-11-05/laposte_hexasmal.csv` |
| INSEE, Code officiel géographique (COG) 2026 | `insee_cog/2026/v_commune_2026.csv`, `v_departement_2026.csv`, `v_region_2026.csv` |
| INSEE, populations légales 2023 | `insee_populations/2023/donnees_departements.csv` |
| Contours des départements | `contours_departements/2026-11-05/departements-version-simplifiee.geojson` |
| Agreste RA 2020 (déjà préparé) | `data/interim/agreste_ra2020_departements.csv` |

In [1]:
import json
from pathlib import Path

import pandas as pd

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
RAW = RACINE / "data/raw"

def lire(chemin, **options):
    """Lecture brute : tout en texte, rien d'interprété."""
    return pd.read_csv(RAW / chemin, dtype=str, keep_default_na=False, **options)

## 1. Les formats de fichiers : chaque source a ses habitudes

| Fichier | Séparateur | Encodage | Particularité |
|---|---|---|---|
| La Poste | `;` | **ISO-8859-1** | En-tête précédé de `#` |
| INSEE COG | `,` | UTF-8 | Chaque valeur entre **guillemets** `"..."` |
| INSEE populations | `;` | UTF-8 | Fins de ligne Windows (CRLF) |

Dans Talend, chaque `tFileInputDelimited` aura donc son propre réglage (séparateur, encodage, caractère d'entourage `"`).
C'est pour ça qu'on décrit chaque source avant de construire les jobs.

In [2]:
laposte = lire("laposte_codes_postaux/2026-11-05/laposte_hexasmal.csv", sep=";", encoding="latin-1")
laposte.columns = [c.lstrip("#") for c in laposte.columns]

communes = lire("insee_cog/2026/v_commune_2026.csv", sep=",")
departements = lire("insee_cog/2026/v_departement_2026.csv", sep=",")
regions = lire("insee_cog/2026/v_region_2026.csv", sep=",")
populations = lire("insee_populations/2023/donnees_departements.csv", sep=";")

for nom, df in [("La Poste", laposte), ("COG communes", communes), ("COG départements", departements),
                ("COG régions", regions), ("Populations", populations)]:
    print(f"{nom:18} {df.shape}")

La Poste           (39192, 5)
COG communes       (37496, 12)
COG départements   (101, 7)
COG régions        (18, 6)
Populations        (100, 9)


## 2. La Poste : code postal → commune

In [3]:
laposte.head()

,Code_commune_INSEE,Nom_de_la_commune,Code_postal,Libellé_d_acheminement,Ligne_5
0,01001,L ABERGEMENT CLEMENCIAT,01400,L ABERGEMENT CLEMENCIAT,
1,01002,L ABERGEMENT DE VAREY,01640,L ABERGEMENT DE VAREY,
2,01004,AMBERIEU EN BUGEY,01500,AMBERIEU EN BUGEY,
3,01005,AMBERIEUX EN DOMBES,01330,AMBERIEUX EN DOMBES,
4,01006,AMBLEON,01300,AMBLEON,


In [4]:
print("Lignes                   :", len(laposte))
print("Codes postaux distincts  :", laposte["Code_postal"].nunique())
print("Codes communes distincts :", laposte["Code_commune_INSEE"].nunique())
print("Lignes avec Ligne_5      :", (laposte["Ligne_5"] != "").sum())
print("Couples (CP, commune) en double :", laposte.duplicated(["Code_postal", "Code_commune_INSEE"]).sum())

Lignes                   : 39192
Codes postaux distincts  : 6328
Codes communes distincts : 35007
Lignes avec Ligne_5      : 4804
Couples (CP, commune) en double : 3681


Une ligne de La Poste n'est pas « un code postal » mais un **couple code postal × commune**, éventuellement précisé
par un lieu-dit (`Ligne_5`). D'où les doublons de couples : même code postal, même commune, mais deux lieux-dits différents.

**Le grain qui nous intéresse** pour la jointure : un couple (code postal, commune), sans doublon.

La relation est **plusieurs à plusieurs** :

In [5]:
couples = laposte[["Code_postal", "Code_commune_INSEE"]].drop_duplicates()
communes_par_cp = couples.groupby("Code_postal").size()
cp_par_commune = couples.groupby("Code_commune_INSEE").size()
print("Communes par code postal :", communes_par_cp.describe()[["mean", "max"]].round(1).to_dict())
print("Codes postaux par commune:", cp_par_commune.describe()[["mean", "max"]].round(1).to_dict())
print("Exemple : CP qui couvre le plus de communes →", communes_par_cp.idxmax(), "avec", communes_par_cp.max())
print("Exemple : commune avec le plus de CP        →", cp_par_commune.idxmax(), "avec", cp_par_commune.max())

Communes par code postal : {'mean': 5.6, 'max': 46.0}
Codes postaux par commune: {'mean': 1.0, 'max': 6.0}
Exemple : CP qui couvre le plus de communes → 51300 avec 46
Exemple : commune avec le plus de CP        → 31555 avec 6


## 3. INSEE COG : commune → département → région

Le COG décrit la géographie administrative officielle au 1er janvier 2026.
La colonne `TYPECOM` distingue plusieurs types de « communes » :

| `TYPECOM` | Signification |
|---|---|
| `COM` | Commune (la seule qui a un département et une région) |
| `COMD` | Commune déléguée : ancienne commune fusionnée dans une **commune nouvelle** |
| `COMA` | Commune associée (ancien régime de fusion) |
| `ARM` | Arrondissement municipal de Paris, Lyon, Marseille |

Pour `COMD`, `COMA` et `ARM`, la colonne `COMPARENT` donne la commune de rattachement.

In [6]:
communes["TYPECOM"].value_counts()

TYPECOM
COM     34875
COMD     2105
COMA      471
ARM        45
Name: count, dtype: int64

In [7]:
# Les communes de plein exercice ont-elles toutes un département et une région ?
com = communes[communes["TYPECOM"] == "COM"]
print("Communes (COM)           :", len(com))
print("Code commune unique      :", com["COM"].is_unique)
print("Sans département         :", (com["DEP"] == "").sum())
print("Sans région              :", (com["REG"] == "").sum())
communes[communes["TYPECOM"] != "COM"].head(3)

Communes (COM)           : 34875
Code commune unique      : True
Sans département         : 0
Sans région              : 0


,TYPECOM,COM,REG,DEP,CTCD,ARR,TNCC,NCC,NCCENR,LIBELLE,CAN,COMPARENT
14,COMD,01015,,,,,1,ARBIGNIEU,Arbignieu,Arbignieu,,01015
23,COMD,01025,,,,,0,BAGE LA VILLE,Bâgé-la-Ville,Bâgé-la-Ville,,01025
32,COMD,01033,,,,,0,BELLEGARDE SUR VALSERINE,Bellegarde-sur-Valserine,Bellegarde-sur-Valserine,,01033


### Piège : le code commune n'est pas unique dans ce fichier

Une **commune nouvelle** garde souvent le code de l'ancienne commune chef-lieu, qui devient aussi une commune déléguée
avec **le même code**. Le grain du fichier n'est donc pas « un code commune » mais **un couple type × code**.

In [8]:
partages = communes[communes["COM"].duplicated(keep=False)].sort_values(["COM", "TYPECOM"])
print("Codes portés par plusieurs lignes :", partages["COM"].nunique())
partages[["TYPECOM", "COM", "LIBELLE", "COMPARENT"]].head(4)

Codes portés par plusieurs lignes :

 584


,TYPECOM,COM,LIBELLE,COMPARENT
13,COM,01015,Arboys en Bugey,
14,COMD,01015,Arbignieu,01015
22,COM,01025,Bâgé-Dommartin,
23,COMD,01025,Bâgé-la-Ville,01025


Conséquence : pour retrouver le type d'un code, il faut **donner la priorité à la ligne `COM`**.
Prendre « la première ligne trouvée » donnerait parfois `COMD`, au hasard de l'ordre du fichier.
C'est le genre d'erreur silencieuse qu'un contrôle d'unicité de clé détecte.

### Les codes La Poste existent-ils dans le COG ?

C'est le maillon le plus fragile : La Poste et l'INSEE ne mettent pas leurs fichiers à jour au même rythme.
Quand des communes fusionnent en **commune nouvelle**, l'ancien code peut rester chez La Poste,
alors que l'INSEE l'a transformé en commune déléguée.

In [9]:
# Priorité à la ligne COM quand un code est partagé (tri : COM d'abord)
priorite = communes.sort_values("TYPECOM", key=lambda s: s != "COM").drop_duplicates("COM")
type_par_code = priorite.set_index("COM")["TYPECOM"]
codes_laposte = pd.Series(couples["Code_commune_INSEE"].unique())
statut = codes_laposte.map(type_par_code).fillna("ABSENT DU COG")
statut.value_counts()

COM              34872
ABSENT DU COG       90
ARM                 45
Name: count, dtype: int64

In [10]:
# Les codes absents du COG : à quoi ressemblent-ils ?
absents = codes_laposte[statut == "ABSENT DU COG"]
laposte[laposte["Code_commune_INSEE"].isin(absents)].drop_duplicates("Code_commune_INSEE").head(10)

,Code_commune_INSEE,Nom_de_la_commune,Code_postal,Libellé_d_acheminement,Ligne_5
38902,97501,MIQUELON LANGLADE,97500,ST PIERRE ET MIQUELON,LANGLADE
38904,97502,ST PIERRE,97500,ST PIERRE ET MIQUELON,ST PIERRE
38928,97701,ST BARTHELEMY,97133,ST BARTHELEMY,
38929,97801,ST MARTIN,97150,ST MARTIN,
38930,98611,ALO,98610,ALO,
38931,98612,SIGAVE,98620,SIGAVE,
38932,98613,UVEA,98600,UVEA,
38933,98711,ANAA,98760,TUUHORA,ANAA
38936,98712,ARUE,98701,ARUE,
38937,98713,ARUTUA,98761,RAUTINI,ARUTUA


Les codes absents du COG commencent par `975` à `988` : ce sont les **collectivités d'outre-mer**
(Saint-Pierre-et-Miquelon, Saint-Barthélemy, Saint-Martin, Wallis-et-Futuna, Polynésie, Nouvelle-Calédonie).
Elles ne sont pas des départements, et l'INSEE les décrit dans un fichier séparé.
Aucune commune déléguée n'apparaît : La Poste utilise ici des codes à jour.

**Règle de résolution d'un code commune vers une commune de plein exercice :**
- `COM` → la commune elle-même ;
- `COMD`, `COMA`, `ARM` → sa commune parente (`COMPARENT`) ;
- absent du COG → département déduit du code commune (2 premiers caractères, ou 3 pour l'outre-mer).

In [11]:
parent = priorite.set_index("COM")["COMPARENT"]

def commune_de_rattachement(code):
    t = type_par_code.get(code)
    if t == "COM":
        return code
    if t in ("COMD", "COMA", "ARM"):
        return parent[code]
    return None

couples["commune"] = couples["Code_commune_INSEE"].map(commune_de_rattachement)
departement_par_commune = com.set_index("COM")["DEP"]
couples["departement"] = couples["commune"].map(departement_par_commune)
print("Couples résolus vers une commune :", f"{couples['commune'].notna().mean():.1%}")
couples.head()

Couples résolus vers une commune : 99.6%


,Code_postal,Code_commune_INSEE,commune,departement
0,01400,01001,01001,01
1,01640,01002,01002,01
2,01500,01004,01004,01
3,01330,01005,01005,01
4,01300,01006,01006,01


## 4. Départements et régions

In [12]:
print("Départements :", len(departements), "| Régions :", len(regions))
print("Codes non standard :", [d for d in departements["DEP"] if not (d.isdigit() and len(d) == 2)])
regions[["REG", "LIBELLE"]]

Départements : 101 | Régions : 18
Codes non standard : ['2A', '2B', '971', '972', '973', '974', '976']


,REG,LIBELLE
0,01,Guadeloupe
1,02,Martinique
2,03,Guyane
3,04,La Réunion
4,06,Mayotte
5,11,Île-de-France
6,24,Centre-Val de Loire
7,27,Bourgogne-Franche-Comté
8,28,Normandie
9,32,Hauts-de-France


In [13]:
# Chaque département a-t-il une région connue ?
print("Départements avec une région inconnue :",
      (~departements["REG"].isin(regions["REG"])).sum())

Départements avec une région inconnue : 0


## 5. Populations légales 2023

Deux populations sont publiées :
- **PMUN** (population municipale) : les personnes qui habitent dans le département. **C'est la référence statistique**, elle ne compte personne deux fois ;
- **PTOT** (population totale) : PMUN + population « comptée à part » (étudiants, internes…), qui compte certaines personnes dans deux communes.

On utilisera **PMUN** pour la densité d'opérateurs.

In [14]:
populations["PMUN"] = populations["PMUN"].astype(int)
print("Départements :", len(populations))
print(f"Population municipale totale : {populations['PMUN'].sum():,}")
print("Départements du COG absents des populations :",
      sorted(set(departements["DEP"]) - set(populations["DEP"])))

Départements : 100
Population municipale totale : 68,094,280
Départements du COG absents des populations : ['976']


## 6. Contours des départements (GeoJSON)

Un **GeoJSON** est un fichier JSON qui décrit des formes géographiques : chaque *feature* a des propriétés (code, nom)
et une géométrie (polygone de coordonnées longitude / latitude). Power BI s'en sert pour colorer une carte.

In [15]:
geo = json.loads((RAW / "contours_departements/2026-11-05/departements-version-simplifiee.geojson").read_text(encoding="utf-8"))
codes_geo = {f["properties"]["code"] for f in geo["features"]}
print("Départements dessinés :", len(codes_geo))
print("Types de géométrie    :", {f["geometry"]["type"] for f in geo["features"]})
print("Absents des contours  :", sorted(set(departements["DEP"]) - codes_geo))

Départements dessinés : 96
Types de géométrie    : {'Polygon', 'MultiPolygon'}
Absents des contours  : ['971', '972', '973', '974', '976']


Le fichier ne contient que la **métropole** : les 5 départements d'outre-mer n'y sont pas.
Pour les afficher, il faudra un second fichier de contours. C'est une limite à traiter avant la page « carte » de Power BI.

`MultiPolygon` apparaît pour les départements faits de plusieurs morceaux (îles).

## 7. Les codes département sont-ils cohérents entre toutes les sources ?

C'est la **clé de jointure commune** à toute la dimension géographique. On compare les listes de codes.

In [16]:
agreste = pd.read_csv(RACINE / "data/interim/agreste_ra2020_departements.csv", dtype={"code_departement": str})

sources = {
    "COG": set(departements["DEP"]),
    "Populations": set(populations["DEP"]),
    "Agreste": set(agreste["code_departement"]),
    "Contours": codes_geo,
}
reference = sources["COG"]
pd.DataFrame({
    nom: {"départements": len(codes),
          "absents par rapport au COG": ", ".join(sorted(reference - codes)) or "-",
          "inconnus du COG": ", ".join(sorted(codes - reference)) or "-"}
    for nom, codes in sources.items()
}).T

,départements,absents par rapport au COG,inconnus du COG
COG,101,-,-
Populations,100,976,-
Agreste,101,-,-
Contours,96,"971, 972, 973, 974, 976",-


## 8. Test de la chaîne complète sur l'annuaire

On applique les règles de nettoyage décidées dans le notebook de l'annuaire (`trim`, zéro initial), puis on rattache
chaque opérateur à un département.

In [17]:
annuaire = lire("agence_bio/2026-11-05/operateursbio.csv", sep=";")
cp = annuaire["CODE POSTAL SIEGE SOCIAL"].str.strip().str.zfill(5)

# Départements possibles pour chaque code postal
deps_par_cp = couples.dropna(subset=["departement"]).groupby("Code_postal")["departement"].agg(lambda s: sorted(set(s)))
deps = cp.map(deps_par_cp)

un_seul = deps.map(lambda d: isinstance(d, list) and len(d) == 1)
plusieurs = deps.map(lambda d: isinstance(d, list) and len(d) > 1)
inconnu = deps.isna()
print(f"Département unique via La Poste : {un_seul.mean():.2%}")
print(f"Plusieurs départements possibles : {plusieurs.sum()}")
print(f"Code postal inconnu (CEDEX…)    : {inconnu.sum()}")

Département unique via La Poste : 99.12%
Plusieurs départements possibles : 347
Code postal inconnu (CEDEX…)    : 430


Pour les codes postaux inconnus (CEDEX), on déduit le département du code postal lui-même :
- `97x..` → les 3 premiers chiffres (outre-mer) ;
- `20...` → Corse : `200xx` et `201xx` = Corse-du-Sud (`2A`), `202xx` à `206xx` = Haute-Corse (`2B`) ;
- `99...` → ce n'est pas un code postal mais le **code INSEE d'un pays** (`99132` = Royaume-Uni) : siège à l'étranger, pas de département ;
- `980..` → **Monaco**, qui a des codes postaux au format français mais n'est pas en France : étranger aussi ;
- sinon → les 2 premiers chiffres.

In [18]:
def departement_depuis_cp(c: str):
    if c.startswith("99") or c.startswith("980"):
        return None                      # étranger : code INSEE d'un pays (99...) ou Monaco (980..)
    if c.startswith("97"):
        return c[:3]
    if c.startswith("20"):
        return "2A" if c[2] in "01" else "2B"
    return c[:2]

deduit = cp[inconnu].map(departement_depuis_cp)
print("Départements déduits valides (présents dans le COG) :", f"{deduit.isin(reference).mean():.1%}")
print("Sans département (étranger) :", cp[inconnu][deduit.isna()].tolist())
print("Non valides restants        :", cp[inconnu][deduit.notna() & ~deduit.isin(reference)].tolist())
deduit.value_counts().head()

Départements déduits valides (présents dans le COG) : 99.5%
Sans département (étranger) : ['99132', '98000']
Non valides restants        : []


CODE POSTAL SIEGE SOCIAL
92    108
94     31
44     16
29     15
66     14
Name: count, dtype: int64

In [19]:
# Pour les CP à plusieurs départements : le département déduit du CP fait-il partie des candidats ?
candidats = deps[plusieurs]
ok = [departement_depuis_cp(c) in d for c, d in zip(cp[plusieurs], candidats)]
print(f"Département déduit parmi les candidats : {sum(ok)}/{len(ok)}")
pd.DataFrame({"code_postal": cp[plusieurs], "candidats": candidats}).drop_duplicates("code_postal").head()

Département déduit parmi les candidats : 347/347


,code_postal,candidats
13,33220,"[24, 33]"
237,52100,"[51, 52]"
1248,05130,"[04, 05]"
1372,37160,"[37, 86]"
1681,21340,"[21, 71]"


## 9. Synthèse : la dimension géographique

### Règle de rattachement d'un opérateur à un département

1. Nettoyer le code postal (`trim`, compléter à 5 caractères).
2. Chercher les communes de ce code postal chez La Poste, et les ramener à leur commune de plein exercice (COG : `COMD`, `COMA`, `ARM` → `COMPARENT`).
3. Si toutes ces communes sont dans **un seul département**, c'est le département de l'opérateur.
4. Si elles sont dans **plusieurs départements**, ou si le code postal est **inconnu** (CEDEX), on déduit le département du code postal (règles pour la Corse et l'outre-mer).
5. La **commune** n'est retenue que si le code postal n'en couvre qu'une. Sinon, elle reste vide : on ne devine pas.

6. Si le « code postal » commence par `99` (code d'un pays) ou `980` (Monaco) : siège à l'étranger, département vide.

On note pour chaque opérateur la **méthode** utilisée (`laposte_unique`, `deduit_cp`, `etranger`). C'est de la traçabilité, qui permet de mesurer la qualité du rattachement dans les contrôles.

### Grain et clés de `dim_geographie`

- Grain : **le département** (101 lignes), avec sa région, sa population (PMUN) et ses exploitations agricoles.
- Clé : `code_departement` (texte, 2 ou 3 caractères : `01`, `2A`, `971`). Jamais un nombre, sinon `01` devient `1` et `2A` est impossible.

### Limites à traiter

| Limite | Conséquence | Piste |
|---|---|---|
| Mayotte absente des populations 2023 | Pas de densité pour le 976 | Afficher « non disponible », ou prendre une autre source INSEE |
| Contours sans l'outre-mer | Les 5 DROM n'apparaissent pas sur la carte | Ajouter un fichier de contours de l'outre-mer |
| Code commune non unique dans le COG (584 codes partagés `COM` / `COMD`) | Mauvais type si on prend la première ligne | Toujours donner la priorité à `COM`. Contrôle d'unicité de clé en silver |
| Un code postal couvre souvent plusieurs communes | Commune inconnue pour beaucoup d'opérateurs | Analyse au niveau département. SIRENE pourra donner la commune exacte de l'établissement |

### Pour Talend

- 3 réglages de lecture différents : La Poste (`;`, ISO-8859-1, `#`), COG (`,`, UTF-8, guillemets), populations (`;`, UTF-8).
- Jointures en cascade dans un `tMap` : annuaire → La Poste (code postal) → COG (commune, puis commune parente) → départements → régions.
- Règle de déduction du département (Corse, outre-mer) : une **routine Java** réutilisable, testée une fois et appelée partout.